# Quality checks

Runs between silver and gold. Every check returns a failure count, and zero
means it passed. If any check fails the notebook raises, so that gold is never
rebuilt on data that did not pass.

The checks are defined as a view, `quality_check_results`, rather than as cells.
A view is a stored query, so the same definition can be run from a scheduled job
or from CI without opening this notebook, and it exists in exactly one place.

## What is checked, and why each

**Duplicate keys.** The most dangerous failure in this pipeline. A repeated key
does not get dropped by a join, it multiplies rows, and the result looks
plausible unless you know the number to expect. Checked on `time_utc` for the
three national series, and on `time_utc, area_id` for weather, whose grain is one
row per hour per observation point.

**Hourly gaps.** A `LAG` window function compares each hour to the previous one
and reports any interval other than one hour. This is how the missing hour on
2026-06-04 was originally found.

**Implausible values.** Nulls, and values outside bounds set deliberately far
outside reality. Consumption above 30,000 MW against a winter peak near 15,000.
Temperature outside −60 to +45 against Finnish records of −51 and +37. The
purpose is to catch corrupt data, unit errors and parsing failures, not to
adjudicate unusual weather. A bound set at the edge of reality produces false
alarms and trains the reader to ignore the check.

## One documented exception

The gap check excludes `2026-06-04 13:00` UTC. Fingrid is missing the 12:15
reading that day, so silver correctly drops 12:00 as an incomplete hour. The
exclusion is by exact timestamp with the reason in a comment, so that a new gap
anywhere else still fails the check.

Note that the excluded timestamp is 13:00 rather than 12:00: `LAG` flags the row
*after* a gap, because that is the row which finds its predecessor too far back.

A check that is permanently red is not a check. Either the underlying issue is
fixed, or the known exception is named explicitly so that everything else stays
meaningful.

## The bound that was wrong

On its first run this notebook failed with 14 rows in `silver_wind`. The values
turned out to be small negative production figures, between −0.3 and −33 MW,
clustered in calm periods.

They are genuine. Turbines draw power for control systems, heating and yaw
motors when production is near zero, so measured output can be slightly negative,
in the same way that the day-ahead price can be. The original bound of
`wind_mw < 0` assumed otherwise.

The bound was corrected to −500 MW rather than the data being adjusted. Clamping
negatives to zero would have looked tidier and would have falsified the energy
balance: an hour when wind power consumed electricity would have become an hour
when it did nothing.

A quality rule written without domain knowledge produces false alarms. This one
did, on its first run, and the finding is kept here rather than quietly edited
away.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.energy_weather.quality_check_results AS

-- Duplicate keys: a repeated key multiplies rows in any downstream join
SELECT 'silver_consumption' AS table_name, 'duplicate keys' AS check_name, COUNT(*) AS failing_rows
FROM (SELECT time_utc FROM workspace.energy_weather.silver_consumption GROUP BY time_utc HAVING COUNT(*) > 1)
UNION ALL
SELECT 'silver_wind', 'duplicate keys', COUNT(*)
FROM (SELECT time_utc FROM workspace.energy_weather.silver_wind GROUP BY time_utc HAVING COUNT(*) > 1)
UNION ALL
SELECT 'silver_price', 'duplicate keys', COUNT(*)
FROM (SELECT time_utc FROM workspace.energy_weather.silver_price GROUP BY time_utc HAVING COUNT(*) > 1)
UNION ALL
SELECT 'silver_weather', 'duplicate keys', COUNT(*)
FROM (SELECT time_utc, area_id FROM workspace.energy_weather.silver_weather GROUP BY time_utc, area_id HAVING COUNT(*) > 1)

UNION ALL

-- Gaps in the hourly series, excluding one documented source gap
SELECT 'silver_wind', 'hourly gaps', COUNT(*)
FROM (
  SELECT time_utc
  FROM (
    SELECT time_utc, LAG(time_utc) OVER (ORDER BY time_utc) AS previous_time
    FROM workspace.energy_weather.silver_wind
  )
  WHERE timestampdiff(HOUR, previous_time, time_utc) <> 1
    -- Fingrid is missing the 12:15 reading on 2026-06-04, so 12:00 UTC is
    -- dropped as incomplete. Verified against bronze_wind. Excluded so that
    -- any new gap still fails this check.
    AND time_utc <> TIMESTAMP '2026-06-04 13:00:00'
)

UNION ALL

-- Implausible values: bounds are set far outside reality, so that genuine
-- extreme weather never trips them but corrupt data does
SELECT 'silver_consumption', 'implausible values', COUNT(*)
FROM workspace.energy_weather.silver_consumption
WHERE consumption_mw IS NULL OR consumption_mw <= 0 OR consumption_mw > 30000
UNION ALL
SELECT 'silver_wind', 'implausible values', COUNT(*)
FROM workspace.energy_weather.silver_wind
-- Small negative values are genuine: turbines draw auxiliary power for
-- control systems, heating and yaw motors when production is near zero.
-- Observed range is 0 to -33 MW against several GW of installed capacity.
-- The bound catches corruption without rejecting real measurements.
WHERE wind_mw IS NULL OR wind_mw < -500 OR wind_mw > 25000
UNION ALL
SELECT 'silver_price', 'implausible values', COUNT(*)
FROM workspace.energy_weather.silver_price
WHERE price_eur_mwh IS NULL OR price_eur_mwh < -1000 OR price_eur_mwh > 10000
UNION ALL
SELECT 'silver_weather', 'implausible values', COUNT(*)
FROM workspace.energy_weather.silver_weather
WHERE temperature_c IS NULL OR temperature_c < -60 OR temperature_c > 45
   OR wind_speed_ms IS NULL OR wind_speed_ms < 0 OR wind_speed_ms > 60

In [0]:
from pyspark.sql import functions as F

results = spark.table("workspace.energy_weather.quality_check_results")
results.orderBy("table_name", "check_name").show(truncate=False)

failed = results.filter(F.col("failing_rows") > 0)

if failed.count() > 0:
    raise Exception(
        f"{failed.count()} quality check(s) failed. Do not rebuild gold on this data."
    )

print("All quality checks passed.")

In [0]:
%sql
SELECT time_utc, wind_mw
FROM workspace.energy_weather.silver_wind
WHERE wind_mw IS NULL
   OR wind_mw < 0
   OR wind_mw > 25000
ORDER BY time_utc

In [0]:
%sql
SELECT time_utc, COUNT(*) AS rows
FROM workspace.energy_weather.silver_wind
WHERE time_utc = TIMESTAMP '2026-08-22 09:00:00'
GROUP BY time_utc